# 06 — LoRA against partial fine-tuning, on the held-out split

`src/training/finetune.py` and `docs/new_report.md §2` argue that LoRA is the
wrong tool for this model. The argument is a good one and it is **still only an
argument** — the report says so itself: *"kod ve araçlar hazır, hiçbir sayı
henüz ölçülmedi."* This notebook replaces it with a number.

## What is actually at stake

The concern that motivates the question is not sample count. Anti-UAV410's
train split is hundreds of thousands of annotated frames, and notebook 02's
default subset alone is ~60 000 clips. What is small is **scene diversity**:
sixty sequences is sixty backgrounds, one object class, one sensor. Against
that, the fine-tune moves **9.3 M of 13.9 M parameters** in its second stage.
That is the overfitting risk worth measuring — not "is 400 enough", but "does
moving two thirds of the network on sixty scenes generalise better or worse
than moving half a percent of it".

| | partial fine-tune | LoRA (r=16) |
|---|---|---|
| what moves | the head, then the head + image encoder | `B @ A` beside each of those layers' weights |
| trainable | ~9.3 M | ~0.3 M, measured below |
| regularisation | freezing the memory path | the rank itself, plus that same freeze |
| optimiser state | 2 moments over 9.3 M | 2 moments over 0.3 M |
| what ships | a checkpoint | **the same checkpoint** — the adapters are merged before writing |

That last row is what makes this a fair test. `src/training/lora.py` merges
`B @ A` into the base weights and restores the original modules, so the LoRA
run produces an ordinary EdgeTAM state dict: same config shape, same tracker,
same exporter, same engines, same `tools/eval_antiuav.py` invocation. A `k × k`
convolution composed with a `1 × 1` **is** a `k × k` convolution, so the
factorisation is exact for the RepViT trunk too — the standard "LoRA only
reaches `nn.Linear`" objection is an objection to a Linear-only implementation,
not to the method.

## What is held identical

Everything except the two swapped callbacks. Both runs go through
`src/training/schedule.py`: the same stages, the same clips in the same order
from the same seed, the same losses, the same one-cycle schedule, the same
gradient clipping, the same EMA, the same validation slice, the same rule for
keeping a checkpoint. `--method` changes which parameters get a gradient and
how the checkpoint is written. Nothing else.

**One thing is deliberately not held identical: the learning rate.** LoRA
starts from `B = 0`, so its update has further to travel; every published
recipe gives it roughly an order of magnitude more, and matching the fine-tune's
rate would be testing a badly-tuned LoRA rather than LoRA. The rates for both
are in `tools/train_thermal.py:RATES`, in one place, visible.

In [ ]:
# --- Runtime and repo ---------------------------------------------------
import os, sys
from pathlib import Path

REPO   = Path("/content/sam-dedection")
BRANCH = "claude/elegant-mendel-lttgf8"

if not REPO.exists():
    !git clone -q -b {BRANCH} https://github.com/yigitkayabagci/sam-dedection.git {REPO}
!git -C {REPO} fetch -q origin {BRANCH}
!git -C {REPO} checkout -q {BRANCH} && git -C {REPO} merge -q --ff-only origin/{BRANCH}
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!bash scripts/setup_edgetam.sh 2>&1 | tail -3
!pip install -q -r requirements.txt gdown

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python -m unittest tests.test_lora tests.test_schedule 2>&1 | tail -3

## Data and labels

Both runs train on exactly the same clips and the same pseudo-masks, so the
labelling is done once. If notebook 02 already ran in this session, or wrote
its labels to Drive, point `LABELS` at them and nothing is recomputed;
otherwise the cell below labels the subset the same way notebook 02 does.

In [ ]:
DATA_DIR = Path("/content/data")
WORK     = Path("/content/work")
LABELS   = WORK / "labels"
CKPT     = REPO / "checkpoints"; CKPT.mkdir(exist_ok=True)

TRAIN_SEQUENCES, VAL_SEQUENCES, TEST_SEQUENCES = 60, 16, 12
SIZE, CLIP_LEN, CLIP_STRIDE = 512, 8, 2
LABEL_STRIDE = 3
STEPS_PER_EPOCH, VAL_BATCHES = 400, 24
LORA_R = 16
SEED = 0

# Reuse a label store from an earlier session if there is one on Drive.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    mirrored = Path("/content/drive/MyDrive/edgetam-thermal/labels")
    if mirrored.is_dir() and not LABELS.is_dir():
        import shutil
        shutil.copytree(mirrored, LABELS)
        print(f"reused the label store from {mirrored}")
except Exception as exc:
    print(f"no Drive ({type(exc).__name__})")

!python tools/fetch_antiuav410.py --dest {DATA_DIR} --splits train val test

from tools.fetch_antiuav410 import dataset_root, describe, find_splits
splits = find_splits(DATA_DIR)
DATA = dataset_root(splits)
print(describe(splits))

In [ ]:
# --- Labels, if they are not already there ------------------------------
import json
from src.training import frame_shape, list_sequences
from src.training.labels import Gates, REPORT_FILE, Sam2Teacher, label_sequence, summarise

train = list_sequences(DATA, "train")[:TRAIN_SEQUENCES]
val   = list_sequences(DATA, "val")[:VAL_SEQUENCES]
height, width = frame_shape(train[0].frames[0])

def labelled(split, sequence):
    path = LABELS / split / sequence.name / REPORT_FILE
    if not path.is_file():
        return None
    report = json.loads(path.read_text())
    ok = report.get("stride") == LABEL_STRIDE and report.get("frames") == len(sequence)
    return report if ok else None

missing = [(s, seq) for s, seqs in (("train", train), ("val", val))
           for seq in seqs if labelled(s, seq) is None]
print(f"{len(missing)} sequence(s) still need labelling")

if missing:
    import torch
    from tqdm.auto import tqdm
    VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
    teacher = Sam2Teacher("facebook/sam2.1-hiera-large", device="cuda")
    GATES = Gates(teacher_iou=0.7, box_iou=0.6, area=(0.15, 1.3), component=0.8)
    for split, sequence in tqdm(missing, desc="labelling"):
        label_sequence(sequence, teacher, LABELS / split, gates=GATES, zoom=4.0,
                       min_size=128, frame_size=(width, height),
                       stride=LABEL_STRIDE, batch_size=8 if VRAM < 24 else 32)
    del teacher
    import gc; gc.collect(); torch.cuda.empty_cache()

for split, sequences in (("train", train), ("val", val)):
    reports = [labelled(split, s) for s in sequences]
    print(f"\n### {split}\n{summarise([r for r in reports if r])}")

## The two runs

Same command, same budget, one flag apart. Each writes a checkpoint and a JSON
log — wall time, peak memory, trainable parameters, per-epoch validation loss —
so the cost side of the comparison is measured rather than asserted.

The batch size is measured per run rather than fixed. LoRA's frozen base
weights need no gradient or optimiser buffers, so it will usually fit a larger
one, and **that difference is a result**: normalising it away would hide half
of what the method buys. The number of optimiser *steps* is identical either
way, which is the budget that matters for learning.

In [ ]:
%%time
!python tools/train_thermal.py --method finetune \
    --data {DATA} --labels {LABELS} --out checkpoints/edgetam_thermal_512.pt \
    --sequences {TRAIN_SEQUENCES} --val-sequences {VAL_SEQUENCES} \
    --size {SIZE} --clip-len {CLIP_LEN} --clip-stride {CLIP_STRIDE} \
    --steps {STEPS_PER_EPOCH} --val-batches {VAL_BATCHES} --seed {SEED} \
    --json /content/run_finetune.json 2>&1 | tail -25

In [ ]:
%%time
!python tools/train_thermal.py --method lora --lora-r {LORA_R} \
    --data {DATA} --labels {LABELS} --out checkpoints/edgetam_lora_512.pt \
    --sequences {TRAIN_SEQUENCES} --val-sequences {VAL_SEQUENCES} \
    --size {SIZE} --clip-len {CLIP_LEN} --clip-stride {CLIP_STRIDE} \
    --steps {STEPS_PER_EPOCH} --val-batches {VAL_BATCHES} --seed {SEED} \
    --json /content/run_lora.json 2>&1 | tail -25

In [ ]:
# --- What each run cost -------------------------------------------------
runs = {name: json.loads(Path(f"/content/run_{name}.json").read_text())
        for name in ("finetune", "lora")}

print(f"{'':<12}{'trainable':>12}{'batch':>8}{'peak GiB':>10}{'minutes':>9}"
      f"{'best val':>10}")
for name, run in runs.items():
    trainable = run.get("lora_parameters") or run.get("trainable_parameters", 0)
    print(f"{name:<12}{trainable / 1e6:>11.3f}M{run['batch']:>8}"
          f"{run['peak_gib']:>10.1f}{run['seconds'] / 60:>9.0f}"
          f"{run['best_val_loss']:>10.4f}")

share = (runs['lora'].get('lora_parameters', 1)
         / max(runs['finetune'].get('trainable_parameters', 1), 1))
print(f"\nLoRA trains {share:.2%} of what the fine-tune does "
      f"({runs['lora'].get('adapted_layers', 0)} layers adapted, r={LORA_R})")

## The measurement that decides

Validation clip loss chose each checkpoint, so validation cannot also be the
verdict. Everything below runs on **`test`**, which neither run has seen in any
form, through the deployment path: the same `VideoTracker` the Orin runs,
prompted once on the first annotated frame and left to propagate.

Three checkpoints, one command each, identical except for the config:

- `configs/edgetam_512.yaml` — stock EdgeTAM at 512, the floor
- `configs/edgetam_512_thermal.yaml` — the partial fine-tune
- `configs/edgetam_512_lora.yaml` — LoRA, merged

Watch **dropout episodes** as closely as state accuracy. A mean overlap hides
the difference between one 60-frame loss and sixty 1-frame ones, and those are
different failures — the first is memory poisoning, which is the thing this
whole project is trying to fix.

In [ ]:
for label, config in (("stock",     "configs/edgetam_512.yaml"),
                      ("finetune",  "configs/edgetam_512_thermal.yaml"),
                      ("lora",      "configs/edgetam_512_lora.yaml")):
    print(f"\n===== {label} =====")
    !python tools/eval_antiuav.py --data {DATA} --split test --limit {TEST_SEQUENCES} \
        --tracker edgetam --config {config} --mode crop \
        --json /content/test_{label}.json 2>&1 | tail -4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

scores = {name: json.loads(Path(f"/content/test_{name}.json").read_text())["sequences"]
          for name in ("stock", "finetune", "lora")}

def weighted(rows, key):
    frames = sum(r["frames"] for r in rows)
    return sum(r[key] * r["frames"] for r in rows) / max(frames, 1)

def episodes(rows):
    lengths = [n for r in rows for n in r["dropout_lengths"]]
    return len(lengths), sum(lengths), (float(np.median(lengths)) if lengths else 0.0)

print(f"test split, {TEST_SEQUENCES} sequences, never seen by either run\n")
print(f"{'':<12}{'state acc':>11}{'success AUC':>13}{'episodes':>10}"
      f"{'lost frames':>13}{'median len':>12}")
for name, rows in scores.items():
    count, lost, median = episodes(rows)
    print(f"{name:<12}{weighted(rows, 'state_accuracy'):>11.4f}"
          f"{weighted(rows, 'success_auc'):>13.4f}{count:>10}{lost:>13}{median:>12.1f}")

# Per sequence, both methods against the same stock baseline: a method that
# wins on average by rescuing two sequences and breaking three is not the same
# result as one that helps everywhere, and the mean cannot tell them apart.
base = {r["name"]: r["state_accuracy"] for r in scores["stock"]}
names = [r["name"] for r in scores["stock"]]
y = np.arange(len(names))
fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(names) + 1.6))
for offset, name, colour in ((-0.2, "finetune", "#4c72b0"), (0.2, "lora", "#dd8452")):
    lookup = {r["name"]: r["state_accuracy"] for r in scores[name]}
    ax.barh(y + offset, [lookup.get(n, np.nan) - base[n] for n in names],
            height=0.38, label=name, color=colour)
ax.set_yticks(y); ax.set_yticklabels(names, fontsize=7)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("state accuracy − stock  (test split)")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
# --- Keep the verdict ---------------------------------------------------
verdict = {
    "test_sequences": TEST_SEQUENCES,
    "lora_r": LORA_R,
    "steps_per_epoch": STEPS_PER_EPOCH,
    "runs": runs,
    "test": {name: {"state_accuracy": weighted(rows, "state_accuracy"),
                    "success_auc": weighted(rows, "success_auc"),
                    "episodes": episodes(rows)[0],
                    "lost_frames": episodes(rows)[1]}
             for name, rows in scores.items()},
}
(WORK / "lora_vs_finetune.json").write_text(json.dumps(verdict, indent=2) + "\n")
print(json.dumps(verdict["test"], indent=2))

## How to read it

| what the test split says | what it means, and what to do |
|---|---|
| fine-tune ahead on state accuracy **and** on episodes | the repo's argument holds. Record the LoRA numbers as the measurement that settles it and move on to notebook 03 |
| LoRA ahead, or level at a fraction of the parameters | the regularisation argument was real. Prefer LoRA and raise `TRAIN_SEQUENCES` for the fine-tune before concluding it is beaten — the gap should close as scenes are added |
| both barely above stock | the bottleneck is upstream of the method: too few scenes, or too little mask supervision. Raise `TRAIN_SEQUENCES` and lower `LABEL_STRIDE` before touching either recipe |
| val improves for both, test does not | classic overfitting to sixty scenes. This is exactly the case the comparison exists to catch, and the fix is data, not rank |

**One run is one sample.** Both methods take `--seed`; re-run each with two or
three seeds before believing a gap under ~0.01 state accuracy. The per-sequence
chart is the honest view — a method that wins the mean by rescuing two
sequences and breaking three is not the same result as one that helps
everywhere.

**A tie is a result.** If LoRA matches the fine-tune, it does so while training
~3 % of the parameters, holding a smaller optimiser state, and fitting a larger
batch — and it merges to the same deployment artefact either way. That is the
case where the report's §2 needs rewriting, not the numbers.